In [ ]:
```xml
<VSCode.Cell language="markdown">
# 🗄️ Database Migration Patterns Guide

**Zero-downtime migrations, blue-green strategies, data sync, and cutover strategies**

This guide covers:
- Blue-green database migration
- Shadow data patterns
- Dual-write strategies
- Data validation
- Cutover strategies
- Rollback procedures
- Performance optimization
- Compliance during migration
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1. Blue-Green Database Migration

### Migration Framework
```python
from dataclasses import dataclass, field
from typing import Dict, Optional, List, Any, Callable
from datetime import datetime
from enum import Enum

class MigrationStatus(Enum):
    IDLE = "idle"
    SYNCING = "syncing"
    VALIDATING = "validating"
    READY_CUTOVER = "ready_cutover"
    CUTTING_OVER = "cutting_over"
    COMPLETED = "completed"
    ROLLED_BACK = "rolled_back"
    FAILED = "failed"

@dataclass
class DatabaseConfig:
    """Database configuration"""
    host: str
    port: int
    database: str
    username: str
    password: str = field(repr=False)
    connection_pool_size: int = 10

@dataclass
class MigrationRecord:
    """Track migration progress"""
    migration_id: str
    source_db: DatabaseConfig
    target_db: DatabaseConfig
    status: MigrationStatus
    start_time: datetime
    last_update: datetime
    records_migrated: int = 0
    errors: List[str] = field(default_factory=list)
    validation_results: Dict = field(default_factory=dict)

class BlueBGreenMigration:
    """Blue-green database migration"""
    
    def __init__(self, migration_id: str, source_db: DatabaseConfig, 
                 target_db: DatabaseConfig):
        self.migration_id = migration_id
        self.source_db = source_db
        self.target_db = target_db
        self.record = MigrationRecord(
            migration_id=migration_id,
            source_db=source_db,
            target_db=target_db,
            status=MigrationStatus.IDLE,
            start_time=datetime.utcnow(),
            last_update=datetime.utcnow()
        )
        self.sync_log: List[Dict] = []
    
    def start_migration(self) -> bool:
        """Start the migration process"""
        
        self.record.status = MigrationStatus.SYNCING
        self.record.last_update = datetime.utcnow()
        
        return True
    
    def full_sync(self, tables: List[str], batch_size: int = 1000) -> bool:
        """Perform full data sync"""
        
        for table in tables:
            try:
                rows = self._read_from_source(table)
                
                # Batch insert to target
                for i in range(0, len(rows), batch_size):
                    batch = rows[i:i + batch_size]
                    self._write_to_target(table, batch)
                    self.record.records_migrated += len(batch)
                
                self.sync_log.append({
                    'timestamp': datetime.utcnow().isoformat(),
                    'table': table,
                    'rows': len(rows),
                    'status': 'success'
                })
            
            except Exception as e:
                self.record.errors.append(f"Error syncing {table}: {str(e)}")
                return False
        
        return True
    
    def validate_sync(self) -> Dict:
        """Validate data consistency"""
        
        self.record.status = MigrationStatus.VALIDATING
        
        validation_results = {
            'timestamp': datetime.utcnow().isoformat(),
            'checks': {}
        }
        
        # Row count comparison
        source_counts = self._get_table_counts(self.source_db)
        target_counts = self._get_table_counts(self.target_db)
        
        for table, source_count in source_counts.items():
            target_count = target_counts.get(table, 0)
            match = source_count == target_count
            
            validation_results['checks'][table] = {
                'source_count': source_count,
                'target_count': target_count,
                'match': match
            }
        
        self.record.validation_results = validation_results
        
        # All tables match?
        all_match = all(c['match'] for c in validation_results['checks'].values())
        
        if all_match:
            self.record.status = MigrationStatus.READY_CUTOVER
        
        return validation_results
    
    def _read_from_source(self, table: str) -> List[Dict]:
        """Read data from source database"""
        # Mock implementation
        return [{'id': i, 'data': f'record_{i}'} for i in range(100)]
    
    def _write_to_target(self, table: str, rows: List[Dict]) -> bool:
        """Write data to target database"""
        # Mock implementation
        return True
    
    def _get_table_counts(self, db: DatabaseConfig) -> Dict[str, int]:
        """Get record counts from database"""
        # Mock implementation
        return {'users': 1000, 'orders': 5000}

class DualWriteMigration:
    """Dual-write strategy for migration"""
    
    def __init__(self):
        self.primary_db: Optional[DatabaseConfig] = None
        self.secondary_db: Optional[DatabaseConfig] = None
        self.write_log: List[Dict] = []
        self.dual_write_enabled = False
    
    def enable_dual_write(self, primary: DatabaseConfig, secondary: DatabaseConfig) -> bool:
        """Enable dual-write mode"""
        
        self.primary_db = primary
        self.secondary_db = secondary
        self.dual_write_enabled = True
        
        return True
    
    def write(self, table: str, data: Dict) -> bool:
        """Write to both primary and secondary"""
        
        results = {}
        
        # Write to primary
        try:
            results['primary'] = self._write_to_db(self.primary_db, table, data)
        except Exception as e:
            results['primary'] = False
            self.write_log.append({
                'timestamp': datetime.utcnow().isoformat(),
                'type': 'error',
                'target': 'primary',
                'error': str(e)
            })
        
        # Write to secondary if dual-write enabled
        if self.dual_write_enabled:
            try:
                results['secondary'] = self._write_to_db(self.secondary_db, table, data)
            except Exception as e:
                results['secondary'] = False
                self.write_log.append({
                    'timestamp': datetime.utcnow().isoformat(),
                    'type': 'error',
                    'target': 'secondary',
                    'error': str(e)
                })
        
        # Log write
        self.write_log.append({
            'timestamp': datetime.utcnow().isoformat(),
            'table': table,
            'primary_success': results.get('primary', False),
            'secondary_success': results.get('secondary', False)
        })
        
        # Primary write must succeed
        return results.get('primary', False)
    
    def _write_to_db(self, db: DatabaseConfig, table: str, data: Dict) -> bool:
        """Write to specific database"""
        # Mock implementation
        return True
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 2. Shadow Data & Cutover

### Shadow Processing
```python
class ShadowDataStrategy:
    """Run production traffic against target database"""
    
    def __init__(self, primary_db: DatabaseConfig, shadow_db: DatabaseConfig):
        self.primary_db = primary_db
        self.shadow_db = shadow_db
        self.shadow_requests: List[Dict] = []
        self.discrepancies: List[Dict] = []
        self.shadow_enabled = False
    
    def enable_shadow(self) -> bool:
        """Enable shadow traffic"""
        
        self.shadow_enabled = True
        return True
    
    def execute_query(self, query: str, params: Dict = None) -> Dict:
        """Execute query with shadow"""
        
        # Execute on primary (return results)
        primary_result = self._execute_on_primary(query, params)
        
        # Execute on shadow (compare)
        if self.shadow_enabled:
            shadow_result = self._execute_on_shadow(query, params)
            
            # Compare results
            if primary_result != shadow_result:
                self.discrepancies.append({
                    'timestamp': datetime.utcnow().isoformat(),
                    'query': query,
                    'primary_result': primary_result,
                    'shadow_result': shadow_result
                })
        
        self.shadow_requests.append({
            'timestamp': datetime.utcnow().isoformat(),
            'query': query,
            'shadow_enabled': self.shadow_enabled
        })
        
        return primary_result
    
    def _execute_on_primary(self, query: str, params: Dict) -> Dict:
        """Execute on primary database"""
        # Mock implementation
        return {'status': 'success', 'rows': []}
    
    def _execute_on_shadow(self, query: str, params: Dict) -> Dict:
        """Execute on shadow database"""
        # Mock implementation
        return {'status': 'success', 'rows': []}
    
    def get_discrepancies(self) -> List[Dict]:
        """Get shadow discrepancies"""
        return self.discrepancies

class CutoverStrategy:
    """Cutover from blue to green"""
    
    def __init__(self, migration: BlueBGreenMigration):
        self.migration = migration
        self.cutover_started = False
        self.cutover_completed = False
        self.cutover_log: List[Dict] = []
    
    def dry_run_cutover(self) -> bool:
        """Test cutover without committing"""
        
        # Test routing switch
        self.cutover_log.append({
            'timestamp': datetime.utcnow().isoformat(),
            'event': 'dry_run_started'
        })
        
        # Simulate switch
        time.sleep(1)
        
        # Verify functionality
        is_healthy = self._health_check(self.migration.target_db)
        
        self.cutover_log.append({
            'timestamp': datetime.utcnow().isoformat(),
            'event': 'dry_run_completed',
            'health': is_healthy
        })
        
        return is_healthy
    
    def execute_cutover(self, rollback_on_failure: bool = True) -> bool:
        """Execute cutover"""
        
        self.migration.record.status = MigrationStatus.CUTTING_OVER
        self.cutover_started = True
        
        try:
            # 1. Drain connections from source
            self._drain_connections(self.migration.source_db)
            
            self.cutover_log.append({
                'timestamp': datetime.utcnow().isoformat(),
                'event': 'connections_drained'
            })
            
            # 2. Switch routing
            self._switch_routing(self.migration.target_db)
            
            self.cutover_log.append({
                'timestamp': datetime.utcnow().isoformat(),
                'event': 'routing_switched'
            })
            
            # 3. Health checks
            if not self._health_check(self.migration.target_db):
                raise Exception("Health check failed on target")
            
            self.cutover_log.append({
                'timestamp': datetime.utcnow().isoformat(),
                'event': 'health_check_passed'
            })
            
            # 4. Verify no errors
            if self.migration.record.errors:
                raise Exception("Errors found in migration")
            
            self.migration.record.status = MigrationStatus.COMPLETED
            self.cutover_completed = True
            
            return True
        
        except Exception as e:
            self.cutover_log.append({
                'timestamp': datetime.utcnow().isoformat(),
                'event': 'cutover_failed',
                'error': str(e)
            })
            
            if rollback_on_failure:
                self.rollback()
            
            return False
    
    def rollback(self) -> bool:
        """Rollback to source database"""
        
        # Switch routing back
        self._switch_routing(self.migration.source_db)
        
        self.migration.record.status = MigrationStatus.ROLLED_BACK
        
        self.cutover_log.append({
            'timestamp': datetime.utcnow().isoformat(),
            'event': 'rollback_completed'
        })
        
        return True
    
    def _drain_connections(self, db: DatabaseConfig) -> bool:
        """Drain active connections"""
        # Mock implementation
        return True
    
    def _switch_routing(self, db: DatabaseConfig) -> bool:
        """Switch routing to database"""
        # Mock implementation
        return True
    
    def _health_check(self, db: DatabaseConfig) -> bool:
        """Check database health"""
        # Mock implementation
        return True
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 3. Data Validation & Sync

### Comprehensive Validation
```python
import time

class DataValidator:
    """Validate data consistency"""
    
    def __init__(self):
        self.validation_results: List[Dict] = []
    
    def validate_row_counts(self, source_db: DatabaseConfig, 
                           target_db: DatabaseConfig, 
                           tables: List[str]) -> Dict:
        """Validate row counts match"""
        
        results = {}
        
        for table in tables:
            source_count = self._count_rows(source_db, table)
            target_count = self._count_rows(target_db, table)
            
            results[table] = {
                'source_count': source_count,
                'target_count': target_count,
                'match': source_count == target_count,
                'difference': abs(source_count - target_count)
            }
        
        return results
    
    def validate_checksums(self, source_db: DatabaseConfig,
                          target_db: DatabaseConfig,
                          table: str) -> bool:
        """Validate data checksums"""
        
        source_checksum = self._calculate_checksum(source_db, table)
        target_checksum = self._calculate_checksum(target_db, table)
        
        match = source_checksum == target_checksum
        
        self.validation_results.append({
            'timestamp': datetime.utcnow().isoformat(),
            'table': table,
            'source_checksum': source_checksum,
            'target_checksum': target_checksum,
            'match': match
        })
        
        return match
    
    def validate_sample_rows(self, source_db: DatabaseConfig,
                            target_db: DatabaseConfig,
                            table: str,
                            sample_size: int = 100) -> Dict:
        """Validate random sample of rows"""
        
        source_rows = self._sample_rows(source_db, table, sample_size)
        target_rows = self._sample_rows(target_db, table, sample_size)
        
        matches = sum(1 for sr, tr in zip(source_rows, target_rows) if sr == tr)
        
        return {
            'sample_size': sample_size,
            'matches': matches,
            'mismatch_percentage': ((sample_size - matches) / sample_size) * 100
        }
    
    def _count_rows(self, db: DatabaseConfig, table: str) -> int:
        """Count rows in table"""
        # Mock implementation
        return 1000
    
    def _calculate_checksum(self, db: DatabaseConfig, table: str) -> str:
        """Calculate table checksum"""
        # Mock implementation
        return "abc123def456"
    
    def _sample_rows(self, db: DatabaseConfig, table: str, size: int) -> List[Dict]:
        """Sample rows from table"""
        # Mock implementation
        return [{'id': i} for i in range(size)]
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 4. Database Migration Checklist

✅ **Planning**
- [ ] Schema analyzed
- [ ] Data volume assessed
- [ ] Downtime tolerance defined
- [ ] Success criteria set
- [ ] Team trained

✅ **Preparation**
- [ ] Target database provisioned
- [ ] Schema replicated
- [ ] Indices created
- [ ] Backup validated
- [ ] Rollback plan ready

✅ **Execution**
- [ ] Initial sync completed
- [ ] Data validated
- [ ] Dual-write enabled
- [ ] Shadow traffic running
- [ ] Discrepancies resolved

✅ **Cutover**
- [ ] Dry run successful
- [ ] Connections drained
- [ ] Routing switched
- [ ] Health checks passed
- [ ] Monitoring active

✅ **Post-Migration**
- [ ] Performance verified
- [ ] Backups taken
- [ ] Monitoring alerts
- [ ] Documentation updated
- [ ] Team debriefing
</VSCode.Cell>
```